# 05 — Séries temporais de metano CAMS73

A média global é ponderada por `cell_area`; o ponto é a célula mais próxima. Médias anuais exigem 12 meses completos. Uma referência empírica e anomalias exigem pelo menos dois anos completos, sem interpolação.

In [ ]:
NOTEBOOK = "05_ch4_timeseries"
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "config/config.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Execute dentro do projeto que contém config/config.yaml.")
sys.path.insert(0, str(ROOT / "notebooks"))
from cams_air_quality.config import load_config
from cams_air_quality import ghg_inversion as ghg
from cams_air_quality.extraction import save_table
from available_ch4 import (inventory, open_level, coverage, global_series,
                           point_series, monthly_series, annual_series,
                           reference_cycle, anomaly_series, selected_area)

CONFIG_PATH = Path(os.environ.get("CAMSAQ_CONFIG", ROOT / "config/config.yaml"))
cfg = load_config(CONFIG_PATH)
LEVEL = 1
START = None
END = None
BASELINE_START = None
BASELINE_END = None
BBOX = None
LAT = -23.5505
LON = -46.6333
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = Path(cfg["root"]) / "reports" / "automatic_notebooks" / NOTEBOOK / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
errors = []
print("Saídas:", OUT)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rows, table = inventory(cfg)
ready = table.loc[table.status == "ready"].to_dict("records") if not table.empty else []
info = coverage(ready)
summary = {"source": "GHG_INVERSION", "level": LEVEL, "requested_start": START,
           "requested_end": END, "bbox": BBOX, "coverage": info,
           "status": "no_data", "files": []}
if not ready:
    print("Sem CH4 processado. Execute camsaq ch4 download ou import, depois process.")
else:
    try:
        with open_level(cfg, LEVEL, start=START, end=END) as (ds, selected, provenance):
            area_ds = selected_area(ds, BBOX)
            global_daily = global_series(area_ds)
            point = point_series(ds, LAT, LON)
            months = monthly_series(global_daily)
            years = annual_series(global_daily)
            summary.update(status="ok", **provenance)
            for name, series in (("global_daily", global_daily),
                                 ("point_daily", point),
                                 ("global_monthly", months),
                                 ("global_annual", years)):
                frame = series.to_dataframe(name="ch4_ppb").reset_index()
                frame.attrs = {**provenance, "series": name, "bbox": BBOX}
                save_table(frame, OUT / f"{name}.csv")
            fig, ax = plt.subplots(figsize=(10, 4))
            global_daily.plot(ax=ax)
            ax.set_title("CH4: média espacial diária (ppb)")
            fig.savefig(OUT / "timeseries.png", dpi=130, bbox_inches="tight")
            plt.close(fig)
            if months.sizes["time"]:
                cycle = months.groupby("time.month").mean("time")
                frame = cycle.to_dataframe(name="ch4_ppb").reset_index()
                frame.attrs = {**provenance, "statistic": "descriptive monthly cycle"}
                save_table(frame, OUT / "monthly_cycle.csv")
                fig, ax = plt.subplots(figsize=(8, 4))
                cycle.plot(ax=ax, marker="o")
                ax.set_title("CH4: ciclo mensal descritivo")
                fig.savefig(OUT / "monthly_cycle.png", dpi=130, bbox_inches="tight")
                plt.close(fig)
            if years.sizes["time"]:
                fig, ax = plt.subplots(figsize=(8, 4))
                years.plot(ax=ax, marker="o")
                ax.set_title("CH4: anos completos")
                fig.savefig(OUT / "annual_mean.png", dpi=130, bbox_inches="tight")
                plt.close(fig)
            complete = ghg.complete_years(ghg.complete_months(global_daily))
            reference_years = complete
            if BASELINE_START and BASELINE_END:
                with open_level(cfg, LEVEL, start=BASELINE_START,
                                end=BASELINE_END) as (base_ds, _, _):
                    base_series = global_series(selected_area(base_ds, BBOX))
                    reference_years = ghg.complete_years(ghg.complete_months(base_series))
                    reference = (reference_cycle(base_series, reference_years)
                                 if len(reference_years) >= 2 else None)
            else:
                reference = (reference_cycle(global_daily, complete)
                             if len(complete) >= 2 else None)
            summary["reference_years"] = reference_years if reference is not None else []
            if reference is not None:
                reference = reference.compute()
                frame = reference.to_dataframe(name="ch4_ppb").reset_index()
                frame["n_years"] = len(reference_years)
                frame.attrs = {**provenance, "reference_years": reference_years}
                save_table(frame, OUT / "reference_cycle.csv")
                monthly_anomaly = anomaly_series(global_daily, reference)
                frame = monthly_anomaly.to_dataframe(name="ch4_ppb").reset_index()
                frame.attrs = {**provenance, "reference_years": reference_years}
                save_table(frame, OUT / "anomaly_monthly.csv")
                fig, ax = plt.subplots(figsize=(10, 4))
                monthly_anomaly.plot(ax=ax)
                ax.set_title("CH4: anomalia mensal (ppb)")
                fig.savefig(OUT / "anomaly_monthly.png", dpi=130, bbox_inches="tight")
                plt.close(fig)
                annual_anomaly = ghg.annual_series(
                    global_daily.groupby("time.month") - reference
                )
                if annual_anomaly.sizes["time"]:
                    frame = annual_anomaly.to_dataframe(name="ch4_ppb").reset_index()
                    frame.attrs = {**provenance, "reference_years": reference_years,
                                   "annual_weighting": "target calendar days"}
                    save_table(frame, OUT / "anomaly_annual.csv")
                    fig, ax = plt.subplots(figsize=(8, 4))
                    annual_anomaly.plot(ax=ax, marker="o")
                    ax.set_title("CH4: anomalia anual (ppb)")
                    fig.savefig(OUT / "anomaly_annual.png", dpi=130, bbox_inches="tight")
                    plt.close(fig)
    except Exception as exc:
        errors.append({"error": str(exc)})
        summary["status"] = "error"
        print("Erro CH4:", exc)
(OUT / "analysis.json").write_text(json.dumps(summary, indent=2, default=str),
                                   encoding="utf-8")
(OUT / "errors.json").write_text(json.dumps(errors, indent=2), encoding="utf-8")
for name in ("timeseries.png", "monthly_cycle.png", "annual_mean.png",
             "anomaly_monthly.png", "anomaly_annual.png"):
    if (OUT / name).is_file():
        display(Image(filename=str(OUT / name)))
print("Erros:", len(errors))

Anomalias são diferenças em ppb em relação à média dos meses calendário dos anos completos de referência. A média anual usa todos os dias do ano; não representa tendência inferencial.